# Sri Lanka Soil Classification 

## Cell 1 — Imports & Warnings

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

All libraries imported successfully!


---
## Cell 2 — Step 1: Load Data

In [2]:
print("=" * 65)
print("  Step 1: Loading Dataset")
print("=" * 65)

df = pd.read_csv("dataset/SL_Soil_Data.csv")

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumn Data Types:")
print(df.dtypes.to_string())
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())
print(f"\nNumeric Summary:")
print(df.describe().to_string())

  Step 1: Loading Dataset
Rows    : 50,000
Columns : 12

Column Data Types:
agro_ecological_zone     object
soil_ph                 float64
nitrogen_N              float64
phosphorus_P            float64
potassium_K             float64
soil_moisture           float64
soil_temp               float64
ambient_temp            float64
humidity                float64
rainfall                float64
altitude                float64
soil_type                object

First 3 rows:
  agro_ecological_zone  soil_ph  nitrogen_N  phosphorus_P  potassium_K  soil_moisture  soil_temp  ambient_temp  humidity  rainfall  altitude                   soil_type
0                  Dry     7.88       30.48         13.60       199.73          49.28      31.03         31.86     89.31      4.30     91.08  Calcic Red Yellow Latosols
1         Intermediate     5.99       42.29         14.25       146.40          26.87      25.81         27.12     86.29     11.80    796.92     Reddish Brown Latosolic
2                 

---
## Cell 3 — Step 2A: Label Encoding (Text to Numbers)

In [3]:
print("=" * 65)
print("  Step 2: Preprocessing")
print("=" * 65)

print("\nLabel Encoding (text -> numbers):")

le_zone = LabelEncoder()
df["agro_ecological_zone_encoded"] = le_zone.fit_transform(
    df["agro_ecological_zone"]
)
print("\n  agro_ecological_zone:")
for cls, num in zip(le_zone.classes_, range(len(le_zone.classes_))):
    print(f"    '{cls}' -> {num}")

le_soil = LabelEncoder()
df["soil_type_encoded"] = le_soil.fit_transform(df["soil_type"])
print("\n  soil_type (Target - 14 classes):")
for cls, num in zip(le_soil.classes_, range(len(le_soil.classes_))):
    print(f"    '{cls}' -> {num}")

  Step 2: Preprocessing

Label Encoding (text -> numbers):

  agro_ecological_zone:
    'Dry' -> 0
    'Dry/Wet' -> 1
    'Intermediate' -> 2
    'Wet' -> 3

  soil_type (Target - 14 classes):
    'Alluvial Soils' -> 0
    'Bog and Half-Bog' -> 1
    'Calcic Red Yellow Latosols' -> 2
    'Grumusols' -> 3
    'Immature Brown Loams' -> 4
    'Low Humic Gley' -> 5
    'Non-Calcic Brown' -> 6
    'Red Yellow Latosols' -> 7
    'Red Yellow Podzolic' -> 8
    'Reddish Brown Earths' -> 9
    'Reddish Brown Latosolic' -> 10
    'Regosols' -> 11
    'Solodized Solonetz' -> 12
    'Solonchaks' -> 13


## Cell 4 — Step 2B & 2C: Features, Target & MinMaxScaler

In [4]:
# 2B — Features & Target
NUMERIC_FEATURES = [
    "soil_ph", "nitrogen_N", "phosphorus_P", "potassium_K",
    "soil_moisture", "soil_temp", "ambient_temp",
    "humidity", "rainfall", "altitude",
]
ALL_FEATURES = NUMERIC_FEATURES + ["agro_ecological_zone_encoded"]
TARGET = "soil_type_encoded"

X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

print(f"Input Features  : {len(ALL_FEATURES)}")
print(f"Target Classes  : {df['soil_type'].nunique()} soil types")

# 2C — MinMaxScaler: numeric -> 0..1
print("\nMinMaxScaler — Normalization (0 to 1):")
scaler = MinMaxScaler()
X[NUMERIC_FEATURES] = scaler.fit_transform(X[NUMERIC_FEATURES])
print("  Normalization complete! Sample:")
print(X.head(3).to_string())

Input Features  : 11
Target Classes  : 14 soil types

MinMaxScaler — Normalization (0 to 1):
  Normalization complete! Sample:
    soil_ph  nitrogen_N  phosphorus_P  potassium_K  soil_moisture  soil_temp  ambient_temp  humidity  rainfall  altitude  agro_ecological_zone_encoded
0  0.696491    0.176217      0.295581     0.287015       0.681231   0.744670      0.739648  0.896545  0.143333  0.102636                             0
1  0.364912    0.252336      0.311277     0.198859       0.336462   0.485870      0.494306  0.841636  0.393333  0.996126                             2
2  0.107018    0.730841      0.210577     0.051690       0.879692   0.405057      0.437371  0.212727  0.631667  0.335540                             3


---
## Cell 5 — Step 3: Train-Test Split (80% / 20%)

In [5]:
print("=" * 65)
print("  Step 3: Train-Test Split  (80% / 20%)")
print("=" * 65)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # proportional split across all classes
)
print(f"Training set  : {X_train.shape[0]:,} rows")
print(f"Testing  set  : {X_test.shape[0]:,} rows")

  Step 3: Train-Test Split  (80% / 20%)
Training set  : 40,000 rows
Testing  set  : 10,000 rows


---
## Cell 6 — Step 4A: Random Forest Training (Tuned)

In [6]:
print("=" * 65)
print("  Step 4: Model Training (Tuned Parameters)")
print("=" * 65)

# max_depth=25        -> limits tree depth to prevent overfitting
# min_samples_split=5 -> reduces noise-driven splits
print("\nRandom Forest (n_estimators=200, max_depth=25) ...")
rf_model = RandomForestClassifier(
    n_estimators=200,       # 200 trees — larger voting pool
    max_depth=25,           # maximum tree depth
    min_samples_split=5,    # minimum samples required to split a node
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_preds)
print(f"  Random Forest Accuracy : {rf_acc * 100:.2f}%")

  Step 4: Model Training (Tuned Parameters)

Random Forest (n_estimators=200, max_depth=25) ...
  Random Forest Accuracy : 99.21%


## Cell 7 — Step 4B: Decision Tree Training

In [7]:
print("\nDecision Tree (max_depth=20) ...")
dt_model = DecisionTreeClassifier(max_depth=20, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
dt_acc   = accuracy_score(y_test, dt_preds)
print(f"  Decision Tree Accuracy : {dt_acc * 100:.2f}%")


Decision Tree (max_depth=20) ...
  Decision Tree Accuracy : 98.27%


---
## Cell 8 — Step 5: Model Evaluation

In [8]:
print("=" * 65)
print("  Step 5: Model Evaluation")
print("=" * 65)

best_model = rf_model if rf_acc >= dt_acc else dt_model
best_preds = rf_preds if rf_acc >= dt_acc else dt_preds
best_name  = "Random Forest" if rf_acc >= dt_acc else "Decision Tree"

print(f"\nBest Model : {best_name}")
print(f"Accuracy   : {max(rf_acc, dt_acc)*100:.2f}%")
print("\nClassification Report (per soil type):")
print(classification_report(y_test, best_preds,
                             target_names=le_soil.classes_))

  Step 5: Model Evaluation

Best Model : Random Forest
Accuracy   : 99.21%

Classification Report (per soil type):
                            precision    recall  f1-score   support

            Alluvial Soils       1.00      1.00      1.00       709
          Bog and Half-Bog       1.00      1.00      1.00       705
Calcic Red Yellow Latosols       0.98      0.98      0.98       704
                 Grumusols       0.99      0.99      0.99       705
      Immature Brown Loams       1.00      1.00      1.00       728
            Low Humic Gley       0.98      0.98      0.98       730
          Non-Calcic Brown       1.00      0.99      0.99       726
       Red Yellow Latosols       0.99      1.00      1.00       706
       Red Yellow Podzolic       1.00      1.00      1.00       708
      Reddish Brown Earths       0.98      0.98      0.98       725
   Reddish Brown Latosolic       1.00      1.00      1.00       709
                  Regosols       1.00      1.00      1.00       716


---
## Cell 9 — Step 6: Feature Importance

In [9]:
print("=" * 65)
print("  Step 6: Feature Importance")
print("=" * 65)

importances = pd.Series(
    best_model.feature_importances_, index=ALL_FEATURES
).sort_values(ascending=False)

print("\nFeature Importance (max=1.0) — most influential features for soil type prediction:")
for feat, imp in importances.items():
    bar = "#" * int(imp * 50)
    print(f"  {feat:<35} {imp:.4f}  {bar}")

  Step 6: Feature Importance

Feature Importance (max=1.0) — most influential features for soil type prediction:
  potassium_K                         0.2472  ############
  nitrogen_N                          0.2195  ##########
  soil_ph                             0.1660  ########
  phosphorus_P                        0.1262  ######
  agro_ecological_zone_encoded        0.1125  #####
  rainfall                            0.0620  ###
  ambient_temp                        0.0315  #
  soil_temp                           0.0221  #
  altitude                            0.0091  
  soil_moisture                       0.0020  
  humidity                            0.0020  


---
## Cell 10 — Step 7: Save Model

In [10]:
print("=" * 65)
print("  Step 7: Save Model to models/ folder")
print("=" * 65)

save_folder = "models"
os.makedirs(save_folder, exist_ok=True)

joblib.dump(best_model, os.path.join(save_folder, "soil_model.pkl"))
joblib.dump(scaler,     os.path.join(save_folder, "scaler.pkl"))
joblib.dump(le_zone,    os.path.join(save_folder, "label_encoder_zone.pkl"))
joblib.dump(le_soil,    os.path.join(save_folder, "label_encoder_soil.pkl"))

print(f"Saved: {save_folder}/soil_model.pkl")
print(f"Saved: {save_folder}/scaler.pkl")
print(f"Saved: {save_folder}/label_encoder_zone.pkl")
print(f"Saved: {save_folder}/label_encoder_soil.pkl")

  Step 7: Save Model to models/ folder
Saved: models/soil_model.pkl
Saved: models/scaler.pkl
Saved: models/label_encoder_zone.pkl
Saved: models/label_encoder_soil.pkl


---
## Cell 11 — Live Prediction Helper Functions

In [11]:
def predict_soil(sensor_data: dict) -> dict:
    """
    Takes a raw sensor dictionary, normalizes it, and returns a prediction.
    Use this function for all live system predictions.

    Parameters
    ----------
    sensor_data : dict
        keys: soil_ph, nitrogen_N, phosphorus_P, potassium_K,
              soil_moisture, soil_temp, ambient_temp, humidity,
              rainfall, altitude, agro_ecological_zone (text)

    Returns
    -------
    dict  ->  { "soil_type": str, "confidence_pct": float,
               "top3": list[tuple] }
    """
    # Step 1: encode zone
    zone_enc = le_zone.transform([sensor_data["agro_ecological_zone"]])[0]

    # Step 2: normalize numeric values (REQUIRED for live system)
    raw_num = np.array([[
        sensor_data["soil_ph"],
        sensor_data["nitrogen_N"],
        sensor_data["phosphorus_P"],
        sensor_data["potassium_K"],
        sensor_data["soil_moisture"],
        sensor_data["soil_temp"],
        sensor_data["ambient_temp"],
        sensor_data["humidity"],
        sensor_data["rainfall"],
        sensor_data["altitude"],
    ]])
    scaled_num = scaler.transform(raw_num)

    # Step 3: build full feature vector
    features = np.append(scaled_num[0], zone_enc).reshape(1, -1)

    # Step 4: predict and get probabilities
    pred_enc   = best_model.predict(features)[0]
    soil_name  = le_soil.inverse_transform([pred_enc])[0]
    probas     = best_model.predict_proba(features)[0]
    confidence = probas.max() * 100

    # Top-3 candidates
    top3_idx = np.argsort(probas)[::-1][:3]
    top3 = [(le_soil.classes_[i], round(probas[i]*100, 1))
            for i in top3_idx]

    return {
        "soil_type":      soil_name,
        "confidence_pct": round(confidence, 1),
        "top3":           top3,
    }


def print_prediction(label: str, sensor_data: dict):
    result = predict_soil(sensor_data)
    print(f"\n  [{label}]")
    print(f"     Input  : pH={sensor_data['soil_ph']}, "
          f"K={sensor_data['potassium_K']}, "
          f"N={sensor_data['nitrogen_N']}, "
          f"Rainfall={sensor_data['rainfall']}, "
          f"Zone={sensor_data['agro_ecological_zone']}")
    print(f"  Predicted   : {result['soil_type']}")
    print(f"  Confidence  : {result['confidence_pct']}%")
    print(f"  Top-3 Votes :")
    for rank, (name, pct) in enumerate(result["top3"], 1):
        bar = "#" * int(pct / 2)
        print(f"       {rank}. {name:<35} {pct:5.1f}%  {bar}")


print("predict_soil() and print_prediction() functions loaded successfully!")

predict_soil() and print_prediction() functions loaded successfully!


---
## Cell 12 — Live Prediction Demo: Distinct Sample 1 (Bog & Half-Bog)

In [12]:
print("=" * 65)
print("  Live Prediction Demo")
print("=" * 65)

print("""
Live System Rule:
    Raw sensor data -> scaler.transform() -> model.predict()
    Skipping normalization will cause incorrect predictions.
    The model was trained on the 0..1 range.
""")

# Very low pH + very high rainfall + wet zone = clear distinct signal
print_prediction(
    "Distinct Data — Bog and Half-Bog",
    {
        "soil_ph":              4.5,
        "nitrogen_N":          120.0,
        "phosphorus_P":         10.0,
        "potassium_K":          60.0,
        "soil_moisture":        65.0,
        "soil_temp":            25.0,
        "ambient_temp":         26.0,
        "humidity":             85.0,
        "rainfall":           3500.0,
        "altitude":             20.0,
        "agro_ecological_zone": "Wet",
    }
)

  Live Prediction Demo

Live System Rule:
    Raw sensor data -> scaler.transform() -> model.predict()
    Skipping normalization will cause incorrect predictions.
    The model was trained on the 0..1 range.


  [Distinct Data — Bog and Half-Bog]
     Input  : pH=4.5, K=60.0, N=120.0, Rainfall=3500.0, Zone=Wet
  Predicted   : Bog and Half-Bog
  Confidence  : 100.0%
  Top-3 Votes :
       1. Bog and Half-Bog                    100.0%  ##################################################
       2. Solonchaks                            0.0%  
       3. Solodized Solonetz                    0.0%  


## Cell 13 — Live Prediction Demo: Distinct Sample 2 (Solonchaks)

In [13]:
# Very high pH + very high K + dry zone = salt soil
print_prediction(
    "Distinct Data — Solonchaks (Salt Soil)",
    {
        "soil_ph":              8.9,
        "nitrogen_N":           10.0,
        "phosphorus_P":          8.0,
        "potassium_K":         430.0,
        "soil_moisture":        18.0,
        "soil_temp":            33.0,
        "ambient_temp":         35.0,
        "humidity":             30.0,
        "rainfall":            800.0,
        "altitude":             10.0,
        "agro_ecological_zone": "Dry",
    }
)


  [Distinct Data — Solonchaks (Salt Soil)]
     Input  : pH=8.9, K=430.0, N=10.0, Rainfall=800.0, Zone=Dry
  Predicted   : Solonchaks
  Confidence  : 57.8%
  Top-3 Votes :
       1. Solonchaks                           57.8%  ############################
       2. Regosols                             27.0%  #############
       3. Reddish Brown Latosolic               7.0%  ###


## Cell 14 — Live Prediction Demo: Ambiguous Sample (Mixed Signals)

In [14]:
# Intermediate values — model may not produce a clear prediction
print_prediction(
    "Ambiguous Data — Mixed Signals (Low Confidence Expected)",
    {
        "soil_ph":              6.5,
        "nitrogen_N":           45.0,
        "phosphorus_P":         12.5,
        "potassium_K":         200.0,
        "soil_moisture":        55.0,
        "soil_temp":            28.5,
        "ambient_temp":         30.0,
        "humidity":             75.0,
        "rainfall":           1800.0,
        "altitude":            150.0,
        "agro_ecological_zone": "Intermediate",
    }
)

print("\n" + "=" * 65)
print("  Pipeline v2 completed successfully!")
print("  Note: If confidence is low, review the Top-3 list")
print("        or consider adding Clay% / Texture features to the model.")
print("=" * 65)


  [Ambiguous Data — Mixed Signals (Low Confidence Expected)]
     Input  : pH=6.5, K=200.0, N=45.0, Rainfall=1800.0, Zone=Intermediate
  Predicted   : Reddish Brown Latosolic
  Confidence  : 54.5%
  Top-3 Votes :
       1. Reddish Brown Latosolic              54.5%  ###########################
       2. Alluvial Soils                       16.0%  ########
       3. Immature Brown Loams                 13.5%  ######

  Pipeline v2 completed successfully!
  Note: If confidence is low, review the Top-3 list
        or consider adding Clay% / Texture features to the model.
